In [2]:
import pandas as pd 

In [10]:
df1=pd.read_csv(r"D:\Airbnb Project\listings.csv")

In [7]:
# To remove completely empty columns 
df=df.dropna(how='all',axis=1)

In [4]:
# To grab all the columns that do not belong to the list of columns, we are about to unpivot/melt.
id_vars=[col for col in df.columns if col not in ['availability_30','availability_60','availability_90','availability_365']]

In [15]:
df.to_csv(r"D:\Airbnb Project\listings_cleaned_checkpoint.csv", index=False)

In [2]:
df=pd.read_csv(r"D:\Airbnb Project\listings_cleaned_checkpoint.csv")

In [15]:
# Before we unpivot the columns, we'd first clean the anchor columns to prevent null values multiplying and filling the data.
fill_values={
    'bathrooms':df['bathrooms'].median(),
    'bedrooms':df['bedrooms'].median(),
    'beds':df['beds'].median()
}
df=df.fillna(value=fill_values)

In [16]:
df['price']=df['price'].str.replace('$','',regex=False).str.replace(',','',regex=False)

In [17]:
df['price']=df['price'].astype(float)

In [18]:
df['price']=df['price'].fillna(df['price'].median())

In [5]:
df_long=pd.melt(
    df,
    id_vars=id_vars,
    value_vars=['availability_30', 'availability_60', 'availability_90', 'availability_365'],
    var_name='availability_window',
    value_name='days_available'
)

In [8]:
df_long[['availability_window','days_available']]

,availability_window,days_available
0,30,6
1,30,9
2,30,3
3,30,29
4,30,2
...,...,...
64423,365,364
64424,365,79
64425,365,330
64426,365,53


In [7]:
df_long['availability_window']=df_long['availability_window'].str.extract(r'(\d+)').astype(int)


In [11]:
df_availability = df_long[['id', 'availability_window', 'days_available']].copy() 

In [41]:
# Saving the lean availability fact table to a CSV
df_availability.to_csv(r"D:\Airbnb Project\fact_availability.csv", index=False)

In [18]:
# Now we'd clear the noise of our base table by removing the availability columns and the over descriptive text columns.
columns_to_drop=['availability_30', 'availability_60', 'availability_90', 'availability_365', # Moved 
                 'listing_url','description','picture_url','host_url','host_thumbnail_url','host_picture_url','host_profile_url' ]
df_base=df.drop(columns=columns_to_drop,errors='ignore')
df_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16107 entries, 0 to 16106
Data columns (total 68 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            16107 non-null  int64  
 1   scrape_id                                     16107 non-null  int64  
 2   last_scraped                                  16107 non-null  object 
 3   source                                        16107 non-null  object 
 4   name                                          16107 non-null  object 
 5   host_id                                       16107 non-null  int64  
 6   host_profile_id                               15982 non-null  float64
 7   host_name                                     15982 non-null  object 
 8   hosts_time_as_user_years                      15982 non-null  float64
 9   hosts_time_as_user_months                     15982 non-null 

In [50]:
# Building the host dimension 

host_descriptive_columns = [
    'host_id', 'host_profile_id', 'host_name', 
    'hosts_time_as_user_years', 'hosts_time_as_user_months',
    'hosts_time_as_host_years', 'hosts_time_as_host_months',
    'host_location', 'host_is_superhost', 'host_identity_verified'
]

In [51]:
df_host=df_base[host_descriptive_columns].drop_duplicates(subset=['host_id']).copy()

In [52]:
print(f"unique hosts : {df_host.shape[0]}")
df_host.head()

unique hosts : 5238


,host_id,host_profile_id,host_name,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_is_superhost,host_identity_verified
0,71615,1.462508e+18,Maria Y Mireia Barcelona4Seasons,16.0,2.0,15.0,2.0,"Barcelona, Spain",f,t
1,90417,1.462509e+18,Marnie,16.0,0.0,15.0,2.0,"Catalonia, Spain",t,t
2,73163,1.462508e+18,Andres,16.0,1.0,15.0,2.0,"Barcelona, Spain",t,t
3,158596,1.462511e+18,Ester,15.0,8.0,15.0,2.0,"Barcelona, Spain",f,t
4,177617,1.462512e+18,Joaquin,15.0,8.0,15.0,2.0,"Barcelona, Spain",t,t


In [19]:
# Just realised putting hosts_time_as column that constantly changes with time is not the best to be put in dimensional table.
host_descriptive_columns = [
    'host_id', 'host_profile_id', 'host_name', 
    'host_location', 'host_is_superhost', 'host_identity_verified'
]

In [54]:
df_host = df_base[host_descriptive_columns].drop_duplicates(subset=['host_id']).copy()

In [55]:
df_host.head()

,host_id,host_profile_id,host_name,host_location,host_is_superhost,host_identity_verified
0,71615,1.462508e+18,Maria Y Mireia Barcelona4Seasons,"Barcelona, Spain",f,t
1,90417,1.462509e+18,Marnie,"Catalonia, Spain",t,t
2,73163,1.462508e+18,Andres,"Barcelona, Spain",t,t
3,158596,1.462511e+18,Ester,"Barcelona, Spain",f,t
4,177617,1.462512e+18,Joaquin,"Barcelona, Spain",t,t


In [57]:
# Removing the unwanted columns
df_host=df_host.drop(columns=['host_profile_id'],errors='ignore')

In [58]:
# Checking for nulls 
print(f"Missing locations: {df_host['host_location'].isnull().sum()}")

Missing locations: 1404


In [61]:
df_host.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5238 entries, 0 to 16105
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   host_id                 5238 non-null   int64 
 1   host_name               5198 non-null   string
 2   host_location           3834 non-null   string
 3   host_is_superhost       5198 non-null   string
 4   host_identity_verified  5198 non-null   string
dtypes: int64(1), string(4)
memory usage: 245.5 KB


In [5]:
# Assigning appropriate data types to str columns to prevent memory overhead and fast operations.
string_cols=['host_name','host_location','host_is_superhost','host_identity_verified']
df_host[string_cols]=df_host[string_cols].astype('string')

In [11]:
# To check for boolean variations for the columns 'host_is_superhost','host_identity_verified', to bring them to a common value.
print('Superhost different values:',df_host['host_is_superhost'].value_counts(dropna=False))
print('Identity Verified:',df_host['host_identity_verified'].value_counts(dropna=False))

Superhost different values: host_is_superhost
f    3839
t    1399
Name: count, dtype: Int64
Identity Verified: host_identity_verified
t    4493
f     745
Name: count, dtype: Int64


In [62]:
df_host.to_csv(r"D:\Airbnb Project\dim_host_checkpoint.csv", index=False)

In [3]:
df_host=pd.read_csv(r"D:\Airbnb Project\dim_host_checkpoint.csv")

In [6]:
df_host.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5238 entries, 0 to 5237
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   host_id                 5238 non-null   int64 
 1   host_name               5198 non-null   string
 2   host_location           3834 non-null   string
 3   host_is_superhost       5198 non-null   string
 4   host_identity_verified  5198 non-null   string
dtypes: int64(1), string(4)
memory usage: 204.7 KB


In [10]:
df_host['host_is_superhost']=df_host['host_is_superhost'].fillna('f')
df_host['host_identity_verified']=df_host['host_identity_verified'].fillna('f')

In [14]:
pd.set_option('display.max.rows',100)

In [19]:
# Cleaning the host_name and host_location columns. 
clean_name=df_host['host_name'].str.lower().str.strip() 
# 1. The known pseudo nulls.
known_nulls=r'^(na|unknown|not known|none|-|null|no|not)$'
is_known_null=clean_name.str.match(known_nulls,na=False)
# 2. Only punctuations 
is_pure_punctuation=clean_name.str.match(r'^[^a-z0-9]*$',na=False)
# 3. Too short 
is_too_short=clean_name.str.len()<=1 

all_trash_conditions=is_known_null | is_pure_punctuation | is_too_short 
df_host.loc[all_trash_conditions,'host_name']=pd.NA

In [20]:
df_host['host_name']=df_host['host_name'].fillna('Unknown Host')

In [ ]:
# Host dimension is clean and ready to be saved.
df_host.to_csv(r"D:\Airbnb Project\host_dimension.csv", index=False)

In [23]:
# Cleaning the base table. 
df_host=pd.read_csv("D:\Airbnb Project\listings_dim_host.csv")

<>:2: SyntaxWarning: invalid escape sequence '\A'
<>:2: SyntaxWarning: invalid escape sequence '\A'
C:\Users\rishabh hinduja\AppData\Local\Temp\ipykernel_23680\1340170289.py:2: SyntaxWarning: invalid escape sequence '\A'
  df_host=pd.read_csv("D:\Airbnb Project\listings_dim_host.csv")


In [25]:
df_host.columns.tolist()

['host_id',
 'host_name',
 'host_location',
 'host_is_superhost',
 'host_identity_verified']

In [27]:
columns_to_drop=['host_name','host_location','host_is_superhost','host_identity_verified','host_profile_id']
df_base=df_base.drop(columns=columns_to_drop,errors='ignore')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16107 entries, 0 to 16106
Data columns (total 63 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            16107 non-null  int64  
 1   scrape_id                                     16107 non-null  int64  
 2   last_scraped                                  16107 non-null  object 
 3   source                                        16107 non-null  object 
 4   name                                          16107 non-null  object 
 5   host_id                                       16107 non-null  int64  
 6   hosts_time_as_user_years                      15982 non-null  float64
 7   hosts_time_as_user_months                     15982 non-null  float64
 8   hosts_time_as_host_years                      15982 non-null  float64
 9   hosts_time_as_host_months                     15982 non-null 

In [30]:
# Building host fact table. 
df_base[['scrape_id','last_scraped']]

,scrape_id,last_scraped
0,20260321160757,2026-03-22
1,20260321160757,2026-03-21
2,20260321160757,2026-03-21
3,20260321160757,2026-03-22
4,20260321160757,2026-03-22
...,...,...
16102,20260321160757,2026-03-21
16103,20260321160757,2026-03-21
16104,20260321160757,2026-03-21
16105,20260321160757,2026-03-21


In [3]:
df_base.to_csv(r"D:\Airbnb Project\base_checkpoint.csv")

NameError: name 'df_base' is not defined

In [4]:
df_base=pd.read_csv(r"D:\Airbnb Project\base_checkpoint.csv")

In [5]:
df_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16107 entries, 0 to 16106
Data columns (total 64 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   Unnamed: 0                                    16107 non-null  int64  
 1   id                                            16107 non-null  int64  
 2   scrape_id                                     16107 non-null  int64  
 3   last_scraped                                  16107 non-null  object 
 4   source                                        16107 non-null  object 
 5   name                                          16107 non-null  object 
 6   host_id                                       16107 non-null  int64  
 7   hosts_time_as_user_years                      15982 non-null  float64
 8   hosts_time_as_user_months                     15982 non-null  float64
 9   hosts_time_as_host_years                      15982 non-null 

In [6]:
df_base.columns.tolist()

['Unnamed: 0',
 'id',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'host_id',
 'hosts_time_as_user_years',
 'hosts_time_as_user_months',
 'hosts_time_as_host_years',
 'hosts_time_as_host_months',
 'host_about',
 'host_listings_count',
 'host_has_profile_pic',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'price_quote_checkin_date',
 'price_quote_checkout_date',
 'price_quote_total_price',
 'price_quote_price_per_night',
 'price_quote_raw',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'has_availability',
 'calendar_last_scraped',
 'number_of_reviews',
 'number_of_reviews_ltm',
 'number_of_reviews_l30d',
 'availability_eoy',
 'number_of_reviews_ly',
 'estimated

In [7]:
# Making host fact table
host_fact_columns=['host_id', 
                   'last_scraped',
                   'hosts_time_as_user_years',
                   'hosts_time_as_user_months',
                   'hosts_time_as_host_years',
                   'hosts_time_as_host_months',
                   'host_listings_count',
                   'calculated_host_listings_count',
                   'calculated_host_listings_count_entire_homes',
                   'calculated_host_listings_count_private_rooms',
                   'calculated_host_listings_count_shared_rooms']

In [8]:
df_host_fact=df_base[host_fact_columns].copy()

In [12]:
df_host_fact=df_host_fact.drop_duplicates(subset=['host_id','last_scraped'])

In [16]:
df_host_fact.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5662 entries, 0 to 16105
Data columns (total 11 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   host_id                                       5662 non-null   int64  
 1   last_scraped                                  5662 non-null   object 
 2   hosts_time_as_user_years                      5662 non-null   float64
 3   hosts_time_as_user_months                     5662 non-null   float64
 4   hosts_time_as_host_years                      5662 non-null   float64
 5   hosts_time_as_host_months                     5662 non-null   float64
 6   host_listings_count                           5662 non-null   float64
 7   calculated_host_listings_count                5662 non-null   int64  
 8   calculated_host_listings_count_entire_homes   5662 non-null   int64  
 9   calculated_host_listings_count_private_rooms  5662 non-null   int64

In [15]:
# Filling null values with median..why? coz thats safer than mean in case of outliers and 
# makes way more sense than using 0. 
fill_values={
    'hosts_time_as_user_years':df_host_fact['hosts_time_as_user_years'].median(),
    'hosts_time_as_user_months':df_host_fact['hosts_time_as_user_months'].median(),
    'hosts_time_as_host_years':df_host_fact['hosts_time_as_host_years'].median(),
    'hosts_time_as_host_months':df_host_fact['hosts_time_as_host_months'].median(),
    'host_listings_count':df_host_fact['host_listings_count'].median()
}
df_host_fact=df_host_fact.fillna(value=fill_values)

In [ ]:
df_host_fact.to_csv(r'D:\Airbnb Project\listings_host_.csv')